# Darwin's Gambit - Day/Night Training Loop (Google Colab)

The continual pipeline: **collect** self-play games into an experience pool, then
**train** a GA on that pool (warm-started from the current champion) and **deploy**
the new champion only if it beats the incumbent. Run it day and night; the
deployed champion only ever gets stronger.

Plan: validate the loop on the **scalar** brain (fast), then run the **neural net**
brain (the real grind). Scalar and NN use **separate** pools + registries so the
two experiments never mix.

CPU runtime is fine (this is CPU-parallel, not GPU). Runtime > Change runtime type
> CPU.

### 1. Install dependencies

In [ ]:
!pip -q install chess numpy

### 2. Upload the latest `training.zip` (from your `D:\\chessai` folder)
Rebuilt to include `pipeline.py`, `opponent_model/`, `experience_pool.py`, `deploy.py`.

In [ ]:
from google.colab import files
import zipfile, os, glob, sys
up = files.upload()                       # choose training.zip
with zipfile.ZipFile(next(iter(up))) as z: z.extractall('.')
TRAIN = os.path.dirname(glob.glob('**/pipeline.py', recursive=True)[0])
if TRAIN not in sys.path: sys.path.insert(0, TRAIN)
print('training dir =', TRAIN)

### 3. (Optional but recommended) Persist across sessions with Google Drive
Colab wipes `/content` when the runtime resets. For a multi-day grind, mount Drive
so the pool + champions survive disconnects. **Skip this cell** to keep everything
in `/content` (fine for a single session).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/darwins-gambit'
os.makedirs(BASE, exist_ok=True)
print('persisting under', BASE)

### 4. Wire up streaming + a tiny smoke cycle (~1-2 min)
Proves the loop runs end to end before the long grind.

In [ ]:
import sys, time
sys.stdout.reconfigure(line_buffering=True)   # stream prints in Colab
import pipeline
BASE = globals().get('BASE', '/content')      # Drive path if mounted, else /content

def run(*argv):
    pipeline.main(list(argv))

# One quick scalar cycle into a throwaway dir.
run('--pool', BASE + '/smoke_pool', '--deploy', BASE + '/smoke_deploy',
    '--kind', 'scalar', '--depth', '1', '--seed', '1', '--processes', '2',
    'collect', '--games', '4')
run('--pool', BASE + '/smoke_pool', '--deploy', BASE + '/smoke_deploy',
    '--kind', 'scalar', '--depth', '1', '--seed', '1', '--processes', '2',
    'train', '--population', '6', '--generations', '2', '--gate', '0.5')
print('smoke OK')

### 5. SCALAR day/night loop (fast)
Validates the full collect -> train -> deploy loop and builds a scalar champion.
Each cycle uses a fresh seed so collected games differ. **Stop anytime**
(Runtime > Interrupt) - the deployed champion persists.

In [ ]:
POOL   = BASE + '/pool_scalar'
DEPLOY = BASE + '/deployed_scalar'
CYCLES, GAMES, POP, GENS, DEPTH, GATE = 8, 40, 16, 8, 2, 0.53

for c in range(1, CYCLES + 1):
    t0 = time.time()
    print('\n##### SCALAR CYCLE %d/%d #####' % (c, CYCLES))
    try:
        run('--pool', POOL, '--deploy', DEPLOY, '--kind', 'scalar',
            '--depth', str(DEPTH), '--seed', str(c), '--processes', '2',
            'collect', '--games', str(GAMES))
        run('--pool', POOL, '--deploy', DEPLOY, '--kind', 'scalar',
            '--depth', str(DEPTH), '--seed', str(c), '--processes', '2',
            'train', '--population', str(POP), '--generations', str(GENS),
            '--gate', str(GATE))
    except Exception as e:
        print('cycle error:', repr(e))
    print('cycle %d took %.0fs' % (c, time.time() - t0))

run('--pool', POOL, '--deploy', DEPLOY, 'status')

### 6. NEURAL-NET day/night loop (the real grind)
The same loop on the NN brain (residual: starts ~baseline, evolves upward).
Slower per game, so the numbers are a touch smaller - but let it run **day and
night**; the deployed NN champion keeps improving. Separate pool + registry from
the scalar run, so they never mix.

In [ ]:
POOL   = BASE + '/pool_nn'
DEPLOY = BASE + '/deployed_nn'
CYCLES, GAMES, POP, GENS, DEPTH, GATE = 30, 24, 12, 6, 2, 0.52

for c in range(1, CYCLES + 1):
    t0 = time.time()
    print('\n##### NN CYCLE %d/%d #####' % (c, CYCLES))
    try:
        run('--pool', POOL, '--deploy', DEPLOY, '--kind', 'nn',
            '--depth', str(DEPTH), '--seed', str(c), '--processes', '2',
            'collect', '--games', str(GAMES))
        run('--pool', POOL, '--deploy', DEPLOY, '--kind', 'nn',
            '--depth', str(DEPTH), '--seed', str(c), '--processes', '2',
            'train', '--population', str(POP), '--generations', str(GENS),
            '--gate', str(GATE))
    except Exception as e:
        print('cycle error:', repr(e))
    print('cycle %d took %.0fs' % (c, time.time() - t0))

run('--pool', POOL, '--deploy', DEPLOY, 'status')

### 7. Download the deployed NN champion
Builds `nn_weights.json` (the viewer's format) from the NN registry and downloads
it. Drop that file into `D:\\chessai\\web\\` and the viewer plays your trained net.

In [ ]:
import json
from engine import NNGenome
DEPLOY = BASE + '/deployed_nn'
reg = json.load(open(DEPLOY + '/registry.json'))
champ = reg['versions'][reg['current'] - 1]
print('deploying NN v%d, %.1f%% vs baseline' % (champ['version'], champ['benchmark_winrate'] * 100))
nn = NNGenome.from_vector(champ['vector'])
out = '/content/nn_weights.json'
open(out, 'w').write(json.dumps(nn.to_dict()))
files.download(out)
# Also grab the registries (the full version history) for your records.
files.download(DEPLOY + '/registry.json')